In [2]:
import numpy as np
import pandas as pd
import mne

In [3]:
full_eeg_df = pd.read_csv("thetaEEG_full_2back_demo.csv")
eeg_lsl_starttime = 1783420.5204828 # NOT calibration complete
full_eeg_df["Time"] = full_eeg_df["Time"] + eeg_lsl_starttime
target_timestamp = np.array([1783460.6038521, 1783467.9537836, 1783475.2202884, 1783482.676686 ,1783493.0251028]) # when LIFU Sonicated
pre = 10
post = 10
windows = []
for idx, event in enumerate(target_timestamp): 
    mask = (full_eeg_df['Time'] >= (event - pre)) & (full_eeg_df['Time'] <= (event + post))
    window_df = full_eeg_df.loc[mask].copy()
    window_df["t_rel"] = window_df["Time"] - event 
    window_df["idx"] = idx
    windows.append(window_df)
#windows has 3 dataframes in one -- index them
data = []
for i in range(len(windows)): 
    window = windows[i]
    window = window.drop(columns = ['t_rel', 'idx','Time'])
    data.append(window)
data

[              Ch01          Ch02          Ch03         Ch04      Ch05  \
 7521    459.170349 -18511.677734 -58884.082031 -2937.197998 -200000.0   
 7522    846.695984 -18552.521484 -58562.875000 -2361.035889 -200000.0   
 7523   1014.566589 -18585.662109 -58643.367188 -2637.076904 -200000.0   
 7524    780.940125 -18569.828125 -58683.355469 -2766.299561 -200000.0   
 7525    449.895905 -18520.287109 -58729.199219 -2946.996826 -200000.0   
 ...            ...           ...           ...          ...       ...   
 12516 -5361.414062 -22794.296875  32254.962891 -8136.678223 -200000.0   
 12517 -4531.312500 -22664.906250  32894.355469 -6853.795410 -200000.0   
 12518 -3843.164795 -22901.775391  33922.988281 -4932.213379 -200000.0   
 12519 -4374.385254 -23064.875000  33680.867188 -5450.750000 -200000.0   
 12520 -5342.364746 -22859.527344  32390.025391 -7939.124023 -200000.0   
 
                Ch06           Ch07          Ch08  
 7521   -5561.280273  104176.335938 -13815.762695  
 7522 

In [15]:
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

# The 8 original channels in their physical order on the BCI Core-8
ALL_CHANNEL_MAP = ["Fz", "F4", "FC4", "Cz", "F3", "FCz", "FC5", "F5"]

def process_eeg_pipeline(df_raw, dataset_name="Dataset"):
    """
    Cleans a single EEG DataFrame, removes blink artifacts via automated ICA,
    calculates frequency band powers, and returns a results DataFrame.
    """
    # ---------------------------------------------------------
    # 1. CLEAN RAW DATAFRAME & DYNAMICALLY DROP DEAD CHANNELS
    # ---------------------------------------------------------
    df = df_raw.copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.replace(-200000.0, np.nan)
    
    # Identify which columns are active (not completely dead)
    # Keeping track of indices matching ALL_CHANNEL_MAP
    valid_indices = []
    for col in df.columns:
        # Check if the column is mostly numbers (not entirely NaNs)
        if df[col].dropna().shape[0] > (len(df) * 0.5):
            valid_indices.append(int(col) if isinstance(col, int) or col.isdigit() else df.columns.get_loc(col))
            
    # Keep only the valid columns and interpolate minor drops
    df = df.iloc[:, valid_indices]
    df = df.interpolate(method='linear', limit=5, limit_direction='both')
    df = df.ffill().bfill()
    
    # Construct the channel names list matching ONLY the surviving data columns
    ch_names = [ALL_CHANNEL_MAP[i] for i in valid_indices]
    
    # ---------------------------------------------------------
    # 2. BUILD MNE RAWARRAY
    # ---------------------------------------------------------
    sfreq = 250
    data = df.values.T  # shape: (n_channels, n_samples)
    
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    
    montage = mne.channels.make_standard_montage("standard_1020")
    raw.set_montage(montage)
    
    # ---------------------------------------------------------
    # 4. FILTERING
    # ---------------------------------------------------------
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    raw.notch_filter(60., verbose=False)
    raw.set_eeg_reference('average', verbose=False)
    
    # ---------------------------------------------------------
    # 5. AUTOMATED ICA (No manual input needed)
    # ---------------------------------------------------------
    # Use an integer slightly below max channel count to be safe
    n_comps = min(len(ch_names) - 1, 6)
    ica = mne.preprocessing.ICA(n_components=n_comps, random_state=97, max_iter='auto')
    ica.fit(raw, verbose=False)
    
    # AUTOMATION: Look for components highly correlated with your frontal channels
    # Frontal channels (Fz, F4, F3) capture 90% of a blink's footprint.
    frontal_picks = [ch for ch in ["Fz", "F3", "F4"] if ch in ch_names]
    frontal_picks.append(0)
    
    if frontal_picks:
        # Find the component matching EOG/Blink properties using a frontal channel as proxy
        bad_idx, scores = ica.find_bads_eog(raw, ch_name=frontal_picks[0], verbose=False)
        ica.exclude = bad_idx
    else:
        ica.exclude = [0]  # Fallback to default first component if all else fails
        
    raw_clean = ica.apply(raw.copy(), verbose=False)
    
    # ---------------------------------------------------------
    # 6. PSD & TOTAL BANDPOWER
    # ---------------------------------------------------------
    psds, freqs = raw_clean.compute_psd(fmin=2, fmax=40, n_fft=512, verbose=False).get_data(return_freqs=True)
    
    def band_power_total(psds, freqs, fmin, fmax):
        mask = (freqs >= fmin) & (freqs <= fmax)
        # Using .sum() instead of .mean() captures true integration area
        return psds[:, mask].sum(axis=1)
        
    theta = band_power_total(psds, freqs, 4, 7)
    alpha = band_power_total(psds, freqs, 8, 12)
    beta  = band_power_total(psds, freqs, 13, 30)
    gamma = band_power_total(psds, freqs, 30, 40)
    
    # Calculate Ratios safely
    theta_alpha_ratio = theta / (alpha + 1e-9)
    
    # ---------------------------------------------------------
    # 7. GENERATE RESULTS DATAFRAME FOR THIS FILE
    # ---------------------------------------------------------
    bands_df = pd.DataFrame({
        'Dataset': dataset_name,
        'Channel': ch_names,
        'Theta (4–7 Hz)': theta,
        'Alpha (8–12 Hz)': alpha,
        'Beta (13–30 Hz)': beta,
        'Gamma (30–40 Hz)': gamma,
        'Theta_Alpha_Ratio': theta_alpha_ratio
    })
    
    return bands_df, raw_clean

In [16]:
# Hypothesized dictionary holding your multiple data runs
my_datasets = {f"Trigger_{i}": data[i] for i in range(len(data))}

# Master list to collect all final processed numbers
all_results = []

# THE LOOP
for name, data_df in my_datasets.items():
    print(f"Processing pipeline for: {name}...")
    
    try:
        # Run the single-file pipeline function
        summary_df, clean_mne_obj = process_eeg_pipeline(data_df, dataset_name=name)
        all_results.append(summary_df)
        
        # Optional: Save or plot your clean MNE object here if desired
        # clean_mne_obj.plot_psd()
        
    except Exception as e:
        print(f"❌ Failed to process {name}. Error: {e}")

# Combine every single file's output into one single mega-dataframe
master_results_df = pd.concat(all_results, ignore_index=True)
print("\n--- ALL PROCESSING COMPLETE ---")
master_results_df.head(30)

Processing pipeline for: Trigger_0...
Processing pipeline for: Trigger_1...
Processing pipeline for: Trigger_2...
Processing pipeline for: Trigger_3...
Processing pipeline for: Trigger_4...

--- ALL PROCESSING COMPLETE ---


C:\Users\jshin\AppData\Local\Temp\ipykernel_12904\3621250581.py:71: RuntimeWarning: filter_length (4096) is longer than the signal (2658), distortion is likely. Reduce filter length or filter a longer signal.
  bad_idx, scores = ica.find_bads_eog(raw, ch_name=frontal_picks[0], verbose=False)
C:\Users\jshin\AppData\Local\Temp\ipykernel_12904\3621250581.py:71: RuntimeWarning: filter_length (4096) is longer than the signal (2658), distortion is likely. Reduce filter length or filter a longer signal.
  bad_idx, scores = ica.find_bads_eog(raw, ch_name=frontal_picks[0], verbose=False)


,Dataset,Channel,Theta (4–7 Hz),Alpha (8–12 Hz),Beta (13–30 Hz),Gamma (30–40 Hz),Theta_Alpha_Ratio
0,Trigger_0,Fz,15713.093800,7518.322360,9189.784523,1549.026613,2.089973
1,Trigger_0,F4,47672.445759,15212.523387,10240.897887,907.742672,3.133763
2,Trigger_0,FC4,52676.017930,15896.203018,13165.297035,1553.378594,3.313748
3,Trigger_0,Cz,229304.889792,78083.889188,68625.879870,8386.888113,2.936648
4,Trigger_0,FCz,10359.261517,4443.395281,3828.853787,522.237376,2.331384
5,Trigger_0,FC5,10982.924682,1235.799427,937.109036,85.864873,8.887304
6,Trigger_0,F5,347215.885898,148185.957457,161338.871759,24122.670519,2.343109
7,Trigger_1,Fz,787.200595,70.373639,224.014279,586.256168,11.186015
8,Trigger_1,F4,2411.413392,139.106966,116.122441,237.557020,17.334958
9,Trigger_1,FC4,6290.298226,369.494748,233.932882,740.914300,17.024053
